# Práctica · Semana 5: Representación del conocimiento y razonamiento

**Curso:** Introducción a la Inteligencia Artificial · UDES
**Tiempo estimado:** 2 h 30 min
**Relación con el capítulo:** secciones 5.1 a 5.6

En las semanas anteriores el conocimiento sobre el problema estaba **escondido dentro del código** (en la función sucesor, en la heurística o en la función de evaluación). Esta semana lo sacamos del código y lo escribimos de forma **explícita**, como hechos y reglas que un programa puede leer, combinar y explicar.

| Parte | Qué construyes | Técnicas |
|---|---|---|
| 1 | Un verificador de consecuencia lógica | Lógica proposicional y tablas de verdad |
| 2 a 4 | Un sistema experto que diagnostica fallas de conexión a internet | Base de hechos, base de reglas, encadenamiento hacia adelante y hacia atrás, explicación |
| 5 | Una red semántica de animales | Herencia de propiedades, excepciones y tripletas |
| 6 y 7 | Un termostato difuso | Conjuntos difusos e inferencia de Mamdani; comparación con un termostato de encendido y apagado |

Ejecuta las celdas en orden. Las celdas marcadas con ✏️ son ejercicios. Todas las celdas se probaron con Python 3 y solo usan `numpy` y `matplotlib`, que Google Colab trae instaladas.

In [ ]:
from itertools import product

import matplotlib.pyplot as plt
import numpy as np

## 1. Lógica proposicional: ¿qué se deduce de lo que sé?

Una base de conocimiento (BC) **implica** una sentencia α (se escribe BC ⊨ α) si α es verdadera en **todos** los modelos (asignaciones de verdad) en los que la BC es verdadera (sección 5.1). Con pocas variables se puede comprobar enumerando la tabla de verdad completa.

Representamos cada sentencia como una función de Python que recibe un modelo (un diccionario de variables) y devuelve `True` o `False`.

In [ ]:
def implica(bc, alfa, variables):
    # Comprueba BC ⊨ alfa recorriendo todos los modelos. Devuelve (resultado, contraejemplo).
    for valores in product([True, False], repeat=len(variables)):
        modelo = dict(zip(variables, valores))
        if bc(modelo) and not alfa(modelo):
            return False, modelo                # la BC es verdadera y alfa falsa
    return True, None


# Si llueve, la calle está mojada. Si la calle está mojada, es resbaladiza. Llueve.
variables = ["llueve", "mojada", "resbaladiza"]
bc = lambda m: ((not m["llueve"]) or m["mojada"]) and ((not m["mojada"]) or m["resbaladiza"]) and m["llueve"]

print("BC ⊨ resbaladiza:", implica(bc, lambda m: m["resbaladiza"], variables))
print("BC ⊨ ¬mojada:   ", implica(bc, lambda m: not m["mojada"], variables))

### ✏️ Ejercicio 1. Una falacia frecuente

1. Cambia la base de conocimiento: quita «llueve» y añade «la calle está mojada». ¿Se puede deducir que llueve? Usa `implica` y explica el contraejemplo que devuelve.
2. Esa forma de razonar («si A entonces B; B; por lo tanto A») se llama **afirmación del consecuente**. ¿Por qué un sistema experto de diagnóstico corre el riesgo de cometerla?
3. ¿Cuántos modelos revisa `implica` con 3 variables? ¿Y con 30? ¿Por qué los sistemas reales no razonan con tablas de verdad?

In [ ]:
# Escribe aquí tu nueva base de conocimiento

*Tus respuestas al ejercicio 1:*

## 2. Un sistema experto: base de hechos y base de reglas

Un sistema experto separa el **conocimiento** del **mecanismo de razonamiento** (sección 5.2):

- La **base de hechos** contiene lo que se sabe del caso concreto (las observaciones del usuario y las conclusiones intermedias).
- La **base de reglas** contiene el conocimiento del experto, en forma de reglas **SI** *condiciones* **ENTONCES** *conclusión*.
- El **motor de inferencia** decide qué reglas aplicar y en qué orden.

Nuestro dominio: diagnosticar por qué un equipo no se conecta a internet en casa. Las reglas son una simplificación, escritas como lo haría un técnico de soporte.

In [ ]:
REGLAS = [
    # (nombre, [condiciones], conclusión)
    ("R1",  ["router_sin_luces"],                                         "router_apagado"),
    ("R2",  ["router_apagado"],                                           "RECOMENDAR: encender el router y revisar el cable de corriente"),
    ("R3",  ["router_con_luces", "ningun_dispositivo_conecta"],           "falla_de_red"),
    ("R4",  ["falla_de_red", "luz_internet_roja"],                        "falla_del_proveedor"),
    ("R5",  ["falla_del_proveedor"],                                      "RECOMENDAR: llamar al proveedor de internet"),
    ("R6",  ["falla_de_red", "luz_internet_verde"],                       "falla_del_router"),
    ("R7",  ["falla_del_router"],                                         "RECOMENDAR: reiniciar el router"),
    ("R8",  ["otros_dispositivos_conectan", "equipo_sin_wifi"],           "problema_del_equipo"),
    ("R9",  ["problema_del_equipo", "wifi_desactivado"],                  "RECOMENDAR: activar el wifi del equipo"),
    ("R10", ["problema_del_equipo", "wifi_activado", "red_no_aparece"],   "fuera_de_alcance"),
    ("R11", ["fuera_de_alcance"],                                         "RECOMENDAR: acercarse al router o usar un repetidor"),
    ("R12", ["otros_dispositivos_conectan", "equipo_con_wifi", "paginas_no_cargan"], "problema_de_dns"),
    ("R13", ["problema_de_dns"],                                          "RECOMENDAR: cambiar los servidores DNS"),
    ("R14", ["conexion_lenta", "muchos_dispositivos"],                    "congestion"),
    ("R15", ["congestion"],                                               "RECOMENDAR: pausar descargas y transmisiones en otros equipos"),
]

for nombre, si, entonces in REGLAS[:4]:
    print(f"{nombre}: SI {' Y '.join(si)} ENTONCES {entonces}")

## 3. Encadenamiento hacia adelante

El **encadenamiento hacia adelante** parte de los hechos conocidos y aplica toda regla cuyas condiciones se cumplan, añadiendo su conclusión a la base de hechos, hasta que ya no se pueda deducir nada nuevo (sección 5.3.1). Está **dirigido por los datos**.

En cada ciclo, el motor reúne las reglas aplicables (el **conjunto conflicto**) y elige una con una **estrategia de resolución de conflictos**. Aquí usamos la más simple: la primera regla de la lista que no se haya disparado aún.

El motor guarda qué regla produjo cada hecho, para poder **explicar** sus conclusiones.

In [ ]:
def encadenamiento_adelante(hechos_iniciales, reglas=REGLAS, traza=True):
    hechos = set(hechos_iniciales)
    origen = {h: "dato del usuario" for h in hechos}   # para explicar: quién produjo cada hecho
    disparadas = set()
    ciclo = 0
    while True:
        conflicto = [r for r in reglas
                     if r[0] not in disparadas and all(c in hechos for c in r[1])]
        if not conflicto:
            break
        ciclo += 1
        nombre, si, entonces = conflicto[0]             # resolución: la primera de la lista
        disparadas.add(nombre)
        if traza:
            print(f"Ciclo {ciclo}: conjunto conflicto {[r[0] for r in conflicto]} -> dispara {nombre}: {entonces}")
        if entonces not in hechos:
            hechos.add(entonces)
            origen[entonces] = (nombre, si)
    return hechos, origen


def explicar(hecho, origen, sangria=""):
    # Responde la pregunta «¿cómo llegaste a esa conclusión?».
    o = origen.get(hecho)
    if o == "dato del usuario":
        print(f"{sangria}- {hecho}: lo informó el usuario")
    else:
        nombre, si = o
        print(f"{sangria}- {hecho}: por la regla {nombre}, porque")
        for c in si:
            explicar(c, origen, sangria + "    ")


caso = ["router_con_luces", "ningun_dispositivo_conecta", "luz_internet_roja"]
hechos, origen = encadenamiento_adelante(caso)
recomendaciones = [h for h in hechos if h.startswith("RECOMENDAR")]
print("\nRecomendaciones:", recomendaciones)
print("\n¿Cómo llegaste a esa recomendación?")
explicar(recomendaciones[0], origen)

### ✏️ Ejercicio 2. Probar y ampliar el sistema experto

1. Ejecuta el sistema con estos casos y anota la recomendación y la explicación de cada uno:
   - `otros_dispositivos_conectan`, `equipo_sin_wifi`, `wifi_activado`, `red_no_aparece`
   - `otros_dispositivos_conectan`, `equipo_con_wifi`, `paginas_no_cargan`, `conexion_lenta`, `muchos_dispositivos`
2. En el segundo caso hay dos recomendaciones. ¿En qué orden se dispararon las reglas? ¿Influye la estrategia de resolución de conflictos en las **conclusiones** o solo en el **orden**?
3. Añade **tres reglas nuevas** a partir de tu propia experiencia (por ejemplo, sobre el cable de red, el modo avión o un portal cautivo de un hotel) y pruébalas con un caso.
4. ¿Qué pasa si el usuario informa `luz_internet_roja` y `luz_internet_verde` a la vez? ¿Cómo debería manejar un sistema experto real la información contradictoria?

In [ ]:
caso_1 = ["otros_dispositivos_conectan", "equipo_sin_wifi", "wifi_activado", "red_no_aparece"]
hechos, origen = encadenamiento_adelante(caso_1)
print([h for h in hechos if h.startswith("RECOMENDAR")])

*Tus respuestas al ejercicio 2:*

## 4. Encadenamiento hacia atrás

El **encadenamiento hacia atrás** parte de una **hipótesis** y busca reglas que la concluyan; para cada una, intenta demostrar sus condiciones como nuevas submetas, hasta llegar a hechos que se pueden **preguntar** al usuario (sección 5.3.2). Está **dirigido por los objetivos**: solo pregunta lo que necesita.

Para no depender del teclado, las respuestas del usuario están en un diccionario. Si un dato no está en el diccionario, el programa lo pregunta con `input()`.

In [ ]:
# Hechos que el sistema puede preguntar al usuario (no los concluye ninguna regla)
PREGUNTABLES = {c for _, si, _ in REGLAS for c in si} - {e for _, _, e in REGLAS}


def encadenamiento_atras(meta, respuestas, reglas=REGLAS, conocidos=None, nivel=0, preguntas=None):
    conocidos = {} if conocidos is None else conocidos
    preguntas = [] if preguntas is None else preguntas
    sangria = "    " * nivel
    if meta in conocidos:
        return conocidos[meta]
    if meta in PREGUNTABLES:
        if meta in respuestas:
            valor = respuestas[meta]
        else:
            valor = input(f"¿Es cierto que {meta}? (s/n) ").strip().lower() == "s"
        preguntas.append(meta)
        print(f"{sangria}? {meta} -> {'sí' if valor else 'no'}")
        conocidos[meta] = valor
        return valor
    for nombre, si, entonces in reglas:
        if entonces == meta:
            print(f"{sangria}Para demostrar «{meta}» pruebo {nombre}: SI {' Y '.join(si)}")
            if all(encadenamiento_atras(c, respuestas, reglas, conocidos, nivel + 1, preguntas) for c in si):
                print(f"{sangria}=> «{meta}» demostrado por {nombre}")
                conocidos[meta] = True
                return True
    conocidos[meta] = False
    return False


usuario = {"router_con_luces": True, "ningun_dispositivo_conecta": True,
           "luz_internet_roja": False, "luz_internet_verde": True, "router_sin_luces": False}
preguntas = []
hipotesis = "RECOMENDAR: reiniciar el router"
print("¿Se demuestra la hipótesis?", encadenamiento_atras(hipotesis, usuario, preguntas=preguntas))
print("Preguntas hechas al usuario:", preguntas)

### ✏️ Ejercicio 3. Adelante contra atrás

1. Con las mismas respuestas del usuario, intenta demostrar la hipótesis `"RECOMENDAR: llamar al proveedor de internet"`. ¿Qué preguntas hace? ¿Por qué falla?
2. ¿Cuántas preguntas haría el encadenamiento hacia adelante para llegar a la misma conclusión, si tuviera que preguntar **todos** los datos de antemano? Compáralo con el número de preguntas del encadenamiento hacia atrás.
3. Un médico que sospecha una enfermedad concreta y pide solo los exámenes que la confirmarían, ¿razona hacia adelante o hacia atrás? ¿Y un sistema de alertas que vigila sensores de una fábrica?

In [ ]:
preguntas = []
print(encadenamiento_atras("RECOMENDAR: llamar al proveedor de internet", usuario, preguntas=preguntas))
print("Preguntas:", preguntas)
print("Datos que podría preguntar el sistema:", len(PREGUNTABLES))

*Tus respuestas al ejercicio 3:*

## 5. Redes semánticas: herencia y excepciones

Una **red semántica** representa el conocimiento como un grafo: los nodos son conceptos o individuos y los arcos son relaciones (sección 5.4). La relación más importante es **es_un**, que permite la **herencia**: un concepto hereda las propiedades de sus antepasados, salvo que tenga una propiedad propia que la contradiga (una **excepción**).

La guardamos como una lista de **tripletas** (sujeto, predicado, objeto), el mismo formato que usan las ontologías de la web semántica (RDF).

In [ ]:
TRIPLETAS = [
    ("ave", "es_un", "animal"),          ("mamifero", "es_un", "animal"),
    ("canario", "es_un", "ave"),         ("pinguino", "es_un", "ave"),
    ("murcielago", "es_un", "mamifero"), ("perro", "es_un", "mamifero"),
    ("Piolín", "instancia_de", "canario"), ("Tux", "instancia_de", "pinguino"),
    ("Firulais", "instancia_de", "perro"),
    ("animal", "respira", True),
    ("ave", "tiene", "plumas"),          ("ave", "vuela", True),
    ("mamifero", "tiene", "pelo"),       ("mamifero", "vuela", False),
    ("pinguino", "vuela", False),        ("pinguino", "nada", True),
    ("murcielago", "vuela", True),       ("canario", "color", "amarillo"),
]


def padres(nodo):
    return [o for s, p, o in TRIPLETAS if s == nodo and p in ("es_un", "instancia_de")]


def consultar(nodo, propiedad, camino=None):
    # Busca la propiedad en el nodo y, si no está, la hereda del antepasado más cercano.
    camino = (camino or []) + [nodo]
    propios = [o for s, p, o in TRIPLETAS if s == nodo and p == propiedad]
    if propios:
        return propios[0], camino
    for padre in padres(nodo):
        valor, ruta = consultar(padre, propiedad, camino)
        if valor is not None:
            return valor, ruta
    return None, camino


for nodo, prop in [("Tux", "vuela"), ("Tux", "tiene"), ("Tux", "respira"),
                   ("Piolín", "vuela"), ("Firulais", "vuela"), ("murcielago", "tiene")]:
    valor, ruta = consultar(nodo, prop)
    print(f"¿{nodo} {prop}? -> {valor}    (camino: {' -> '.join(ruta)})")

### ✏️ Ejercicio 4. Ampliar la red

1. Añade el concepto `avestruz` (un ave que no vuela y que corre rápido) y una instancia. Comprueba que la herencia funciona.
2. Añade el concepto `ornitorrinco`: es un mamífero, pero pone huevos. ¿Qué tripletas necesitas?
3. La función `consultar` usa el **antepasado más cercano**. Si un concepto tuviera dos padres con valores contradictorios (por ejemplo, algo que es a la vez `ave` y `mamifero`), ¿qué devolvería? Este es el **problema de la herencia múltiple** (sección 5.4.2).
4. Busca en Wikidata (https://www.wikidata.org) el elemento «penguin» y anota dos de sus relaciones. ¿Qué relación de Wikidata cumple el papel de `es_un`?

In [ ]:
# Escribe aquí tus nuevas tripletas y consultas

*Tus respuestas al ejercicio 4:*

## 6. Lógica difusa: el termostato

Queremos controlar la potencia de un calefactor para mantener una habitación a 22 °C. Un termostato clásico usa reglas **nítidas**: «si la temperatura es menor que 21,5 °C, encender al 100 %; si es mayor que 22,5 °C, apagar». La **lógica difusa** permite escribir reglas con términos vagos, como lo haría una persona: «si hace un poco de frío y la temperatura está bajando, poner potencia media» (secciones 5.5 y 5.6).

### 6.1 Variables lingüísticas y funciones de pertenencia

Usamos dos entradas y una salida:

| Variable | Universo | Etiquetas |
|---|---|---|
| **Error** = objetivo − temperatura actual | −4 a 4 °C | `caliente` (sobra calor), `bien`, `frio` (falta calor) |
| **Tendencia** de la temperatura | −0,5 a 0,5 °C por minuto | `bajando`, `estable`, `subiendo` |
| **Potencia** del calefactor | 0 a 100 % | `nula`, `baja`, `media`, `alta` |

Cada etiqueta es un **conjunto difuso** definido por una **función de pertenencia** triangular o trapezoidal, con valores entre 0 y 1.

In [ ]:
def trap(x, a, b, c, d):
    # Función de pertenencia trapezoidal: 0 antes de a, sube hasta b, 1 entre b y c, baja hasta d.
    x = np.asarray(x, dtype=float)
    sube = np.where(b > a, (x - a) / (b - a + 1e-12), 1.0)
    baja = np.where(d > c, (d - x) / (d - c + 1e-12), 1.0)
    return np.clip(np.minimum(np.minimum(sube, 1.0), baja), 0.0, 1.0)


def tri(x, a, b, c):
    return trap(x, a, b, b, c)


ERROR = {"caliente": lambda x: trap(x, -9, -9, -2, 0),
         "bien":     lambda x: tri(x, -1.5, 0, 1.5),
         "frio":     lambda x: trap(x, 0, 2, 9, 9)}
TENDENCIA = {"bajando":  lambda x: trap(x, -9, -9, -0.2, 0),
             "estable":  lambda x: tri(x, -0.15, 0, 0.15),
             "subiendo": lambda x: trap(x, 0, 0.2, 9, 9)}
POTENCIA = {"nula":  lambda x: trap(x, -1, -1, 0, 20),
            "baja":  lambda x: tri(x, 10, 30, 50),
            "media": lambda x: tri(x, 40, 60, 80),
            "alta":  lambda x: trap(x, 70, 90, 101, 101)}

U_ERROR = np.linspace(-4, 4, 401)
U_TEND = np.linspace(-0.5, 0.5, 401)
U_POT = np.linspace(0, 100, 501)

fig, ejes = plt.subplots(1, 3, figsize=(13, 3))
for ax, (titulo, conjuntos, u) in zip(ejes, [("Error (°C)", ERROR, U_ERROR),
                                              ("Tendencia (°C/min)", TENDENCIA, U_TEND),
                                              ("Potencia (%)", POTENCIA, U_POT)]):
    for etiqueta, mu in conjuntos.items():
        ax.plot(u, mu(u), label=etiqueta)
    ax.set_title(titulo)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 6.2 La base de reglas

Nueve reglas cubren todas las combinaciones de las entradas. Se suelen escribir como una tabla, llamada **matriz de memoria asociativa difusa** (FAM):

| Error \ Tendencia | bajando | estable | subiendo |
|---|---|---|---|
| **frío** | alta | alta | alta |
| **bien** | media | media | baja |
| **caliente** | baja | nula | nula |

In [ ]:
REGLAS_DIFUSAS = [
    # (error, tendencia, potencia)
    ("frio", "bajando", "alta"),     ("frio", "estable", "alta"),     ("frio", "subiendo", "alta"),
    ("bien", "bajando", "media"),    ("bien", "estable", "media"),    ("bien", "subiendo", "baja"),
    ("caliente", "bajando", "baja"), ("caliente", "estable", "nula"), ("caliente", "subiendo", "nula"),
]

### 6.3 Inferencia de Mamdani paso a paso

El método de **Mamdani** (Mamdani y Assilian, 1975; sección 5.6) tiene cuatro pasos:

1. **Fuzzificación:** calcular el grado de pertenencia de cada entrada a cada etiqueta.
2. **Evaluación de las reglas:** el grado de activación de una regla con «Y» es el **mínimo** de los grados de sus condiciones. El conjunto de salida de la regla se **recorta** a esa altura.
3. **Agregación:** unir los conjuntos recortados de todas las reglas con el **máximo**.
4. **Defuzzificación:** convertir el conjunto agregado en un número. Usamos el **centroide**, el centro de gravedad del área.

In [ ]:
def mamdani(error, tendencia, reglas=REGLAS_DIFUSAS, detalle=False):
    # 1. Fuzzificación
    mu_e = {k: float(f(error)) for k, f in ERROR.items()}
    mu_t = {k: float(f(tendencia)) for k, f in TENDENCIA.items()}
    agregado = np.zeros_like(U_POT)
    activaciones = []
    for e, t, p in reglas:
        # 2. Evaluación: Y = mínimo, implicación = recorte
        grado = min(mu_e[e], mu_t[t])
        activaciones.append((e, t, p, grado))
        # 3. Agregación: máximo
        agregado = np.maximum(agregado, np.minimum(grado, POTENCIA[p](U_POT)))
    # 4. Defuzzificación por centroide
    salida = float(np.sum(U_POT * agregado) / np.sum(agregado)) if agregado.sum() > 0 else 0.0
    if detalle:
        return salida, mu_e, mu_t, activaciones, agregado
    return salida


salida, mu_e, mu_t, activ, agregado = mamdani(1.0, -0.1, detalle=True)
print("Entradas: error = 1,0 °C (falta un poco de calor), tendencia = −0,1 °C/min (bajando)")
print("1. Fuzzificación del error:    ", {k: round(v, 2) for k, v in mu_e.items()})
print("   Fuzzificación de la tendencia:", {k: round(v, 2) for k, v in mu_t.items()})
print("2. Reglas activadas:")
for e, t, p, g in activ:
    if g > 0:
        print(f"   SI error es {e} Y tendencia es {t} ENTONCES potencia es {p}: grado {g:.2f}")
print(f"4. Potencia (centroide): {salida:.1f} %")

plt.figure(figsize=(6, 2.8))
for etiqueta, mu in POTENCIA.items():
    plt.plot(U_POT, mu(U_POT), "--", lw=0.8, label=etiqueta)
plt.fill_between(U_POT, agregado, alpha=0.4, label="agregado")
plt.axvline(salida, color="k", label=f"centroide = {salida:.1f}")
plt.legend(fontsize=7)
plt.title("3 y 4. Agregación y defuzzificación")
plt.show()

### 6.4 La superficie de control

La celda calcula la potencia para muchas combinaciones de las entradas. Observa que la salida cambia de forma **gradual**: no hay saltos bruscos como en un termostato nítido.

In [ ]:
E, T = np.meshgrid(np.linspace(-4, 4, 41), np.linspace(-0.5, 0.5, 41))
Z = np.vectorize(mamdani)(E, T)
plt.figure(figsize=(6, 4))
plt.contourf(E, T, Z, levels=20, cmap="coolwarm")
plt.colorbar(label="Potencia (%)")
plt.xlabel("Error (°C): positivo = falta calor")
plt.ylabel("Tendencia (°C/min)")
plt.title("Superficie de control difusa")
plt.show()

## 7. Simulación: termostato difuso contra termostato de encendido y apagado

Simulamos una habitación durante 3 horas, con un paso de un minuto. Afuera hace 10 °C. En cada minuto la habitación pierde el 2 % de su diferencia de temperatura con el exterior, y el calefactor, a plena potencia, aporta 0,4 °C. Para compensar las pérdidas a 22 °C hace falta, por tanto, una potencia del 60 %. A la hora y media se abre una ventana durante 10 minutos y las pérdidas suben al 10 % por minuto. La tendencia se mide como el cambio promedio de la temperatura en los últimos 3 minutos.

El **termostato nítido** tiene una histéresis de ±0,5 °C: enciende al 100 % por debajo de 21,5 °C y apaga por encima de 22,5 °C.

In [ ]:
def simular(controlador, minutos=180, t_inicial=15.0, objetivo=22.0, t_exterior=10.0):
    temperatura, potencia_prev = t_inicial, 0.0
    historia = [temperatura] * 4
    temps, potencias = [], []
    for minuto in range(minutos):
        perdida = 0.10 if 90 <= minuto < 100 else 0.02           # ventana abierta
        tendencia = (historia[-1] - historia[-4]) / 3             # °C por minuto, últimos 3 minutos
        potencia = controlador(objetivo - temperatura, tendencia, potencia_prev)
        temperatura += perdida * (t_exterior - temperatura) + 0.4 * potencia / 100
        historia.append(temperatura)
        potencia_prev = potencia
        temps.append(temperatura)
        potencias.append(potencia)
    return np.array(temps), np.array(potencias)


def nitido(error, tendencia, potencia_prev):
    if error > 0.5:
        return 100.0
    if error < -0.5:
        return 0.0
    return potencia_prev                      # dentro de la banda: mantener el estado


def difuso(error, tendencia, potencia_prev):
    return mamdani(np.clip(error, -4, 4), np.clip(tendencia, -0.5, 0.5))


resultados = {"nítido": simular(nitido), "difuso": simular(difuso)}
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
for nombre, (temps, pots) in resultados.items():
    a1.plot(temps, label=nombre)
    a2.plot(pots, label=nombre)
a1.axhline(22, color="gray", ls=":")
a1.axvspan(90, 100, color="orange", alpha=0.2, label="ventana abierta")
a1.set_ylabel("Temperatura (°C)")
a1.legend()
a2.set_ylabel("Potencia (%)")
a2.set_xlabel("Minuto")
plt.tight_layout()
plt.show()

for nombre, (temps, pots) in resultados.items():
    estable = temps[60:90]                    # tramo estable, antes de abrir la ventana
    cambios = int(np.sum(np.abs(np.diff(pots)) > 5))
    llega = int(np.argmax(temps > 21.5))
    print(f"{nombre:<7} llega a 21,5 °C en el minuto {llega} | media {estable.mean():.2f} °C | "
          f"oscilación ±{(estable.max() - estable.min()) / 2:.2f} °C | cambios bruscos de potencia: {cambios} | "
          f"energía: {pots.sum() / 100:.0f} min a plena potencia")

### ✏️ Ejercicio 5. Ajustar el controlador difuso

1. Compara las dos curvas. ¿Cuál mantiene la temperatura más estable? ¿Cuál enciende y apaga el calefactor más veces? ¿Por qué eso importa en un aparato real? ¿Cuál llega antes a la temperatura deseada?
2. Observa la temperatura media del controlador difuso en el tramo estable. ¿Llega exactamente a 22 °C? Cuando el error es pequeño, la potencia se acerca al 60 % que compensa las pérdidas, pero no llega a superarlo lo suficiente para cerrar la diferencia. Propón un cambio en las reglas o en los conjuntos de salida que reduzca ese error, y pruébalo.
3. Cambia la defuzzificación del centroide por la **media de los máximos** (el promedio de los puntos donde el conjunto agregado alcanza su valor máximo). ¿Cómo cambia la superficie de control?
4. ¿Qué ventaja tiene, para quien mantiene este software, que el controlador esté escrito como nueve reglas legibles en lugar de una fórmula?

In [ ]:
# Espacio para tus cambios: modifica REGLAS_DIFUSAS o POTENCIA y vuelve a ejecutar las secciones 6.3 a 7.

*Tus respuestas al ejercicio 5:*

## 8. Cierre

Completa esta tabla con lo que observaste:

| Representación o técnica | ¿Cómo se escribe el conocimiento? | ¿Cómo se razona? | Una ventaja | Una limitación |
|---|---|---|---|---|
| Lógica proposicional | | | | |
| Sistema experto con encadenamiento hacia adelante | | | | |
| Sistema experto con encadenamiento hacia atrás | | | | |
| Red semántica | | | | |
| Sistema difuso de Mamdani | | | | |

Lleva el cuaderno ejecutado a la sesión sincrónica. Lo que construiste aquí es la base de la parte A de la **Actividad Evaluativa 3**: un sistema basado en reglas o difuso sobre tus propios datos.

---

**Referencias**

García Serrano, A. (2016). *Inteligencia artificial: Fundamentos, práctica y aplicaciones* (2.ª ed.). RC Libros.

Mamdani, E. H., y Assilian, S. (1975). An experiment in linguistic synthesis with a fuzzy logic controller. *International Journal of Man-Machine Studies, 7*(1), 1-13. https://doi.org/10.1016/S0020-7373(75)80002-2

Russell, S. J., y Norvig, P. (2004). *Inteligencia artificial: Un enfoque moderno* (2.ª ed.; J. M. Corchado Rodríguez et al., Trads.). Pearson Educación.

Zadeh, L. A. (1965). Fuzzy sets. *Information and Control, 8*(3), 338-353. https://doi.org/10.1016/S0019-9958(65)90241-X